# Lecture 10 warm-up: beyond the data

**AI-powered Investment and Risk Management (25882), Lecture 10: AI in Risk Forecasting and Stress Testing**

Stress testing asks about months your data have never seen. Machine learning is built to learn from the months it *has* seen. This notebook runs three short experiments on real US data to show what happens where the two meet:

1. **The extrapolation trap.** Push a market shock past history and watch what four models predict for bank stocks.
2. **Crash months are joint.** Compare how 49 industries move together in calm months and in the worst months.
3. **Reverse stress test.** Choose a loss and the history your model learns from, then find the most plausible way to suffer that loss.

Each experiment runs in about two seconds once the setup cell has finished. If the sliders do not appear (for example, if `ipywidgets` is not installed), change the plain variables at the top of each cell and run it again. The same three experiments are in the **Stress Lab** playground, which runs in a browser with nothing to install.

**Data.** Kenneth R. French Data Library (49 industry portfolios and the market factor, CRSP June 2026 vintage) and the Goyal–Welch predictor data to December 2025, all mirrored in the teaching repository. Every number quoted on the Lecture 10 slides comes from these files and is reproduced here.

## Setup

The helper module `airm10.py` holds the plumbing. It finds the data (your local copy of the repository first, then a local cache, then the repository over the internet), loads it, and draws the Plotly figures, so this notebook can stay focused on the ideas. If you opened the notebook on its own, the first lines fetch the helper once; after that everything runs offline.

The `HAVE_WIDGETS` flag decides whether each experiment gets sliders or falls back to the plain variables.

In [ ]:
import os
import urllib.request

if not os.path.exists("airm10.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/VitaliAlexeev/AI_Investments_2026/main/airm10.py", "airm10.py")

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression

import airm10 as A

try:
    import ipywidgets as widgets
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
print("Sliders available:", HAVE_WIDGETS)

## Experiment 1: the extrapolation trap

We model the monthly return of US bank stocks as a function of the market's return, using a **training window** of months. With the default window, 1990 to 2007, the worst month the models ever see is August 1998, when the market fell 15.6%. Then we ask each model about a worse month: exactly the kind of month a stress test is about.

The four models:

- **Linear regression**: a straight line, which keeps going in both directions.
- **Random forest** and **gradient boosting**: built from decision trees, flexible inside the data.
- **Hybrid**: a linear backbone plus gradient boosting on whatever the line misses, with that correction switched off outside the training data. It is one of the three fixes on the slides: keep an economic backbone that extrapolates, and let machine learning work only where there is data.

The red points are months the models were *not* trained on, replayed as historical scenarios: October 1929, October 1987, October 2008 and March 2020.

<details>
<summary><b>The maths: why trees flatline</b></summary>

A regression tree splits the input range into intervals and predicts the average training outcome inside each interval (a *leaf*): $\hat f(x) = \bar y_{\ell(x)}$. Below the lowest split, every $x$ lands in the same leaf, so $\hat f(x)$ is constant there, however extreme $x$ becomes.

A random forest averages $B$ such step functions, each leaf value itself an average of observed outcomes, so

$$\min_x \hat f_{\text{forest}}(x) \;\ge\; \min_i y_i :$$

a forest cannot predict a loss worse than the worst month it has seen, and with at least five months per leaf it usually predicts much less. Gradient boosting adds step functions, so it too is flat beyond the last split.

The hybrid is

$$\hat y = \hat\alpha + \hat\beta x + \hat g(x)\,\mathbf{1}\{x_{\min} \le x \le x_{\max}\},$$

where $\hat g$ is gradient boosting fitted to the residuals of the line. Outside the training range it is just the line.
</details>

The next cell fits the four models on each of four training windows. It is the slowest cell in the notebook (a few seconds); everything after it is fast.

In [ ]:
def fit_models(train):
    X, y = train[["mkt"]].values, train["banks"].values
    return {
        "Linear regression": LinearRegression().fit(X, y),
        "Random forest": RandomForestRegressor(n_estimators=500, min_samples_leaf=5,
                                               random_state=0).fit(X, y),
        "Gradient boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                                           min_samples_leaf=10, random_state=0).fit(X, y),
        "Hybrid": A.HybridRegressor().fit(X, y),
    }

data = A.market_banks()
fitted = {}
for name, (start, end) in A.TRAIN_WINDOWS.items():
    train = data.loc[start:end]
    fitted[name] = (fit_models(train), train)
    print(f"{name}: {len(train)} months; worst market month {train['mkt'].min():.1f}% "
          f"({train['mkt'].idxmin()})")

**Your move.** Drag the shock past the shaded region, then change the training window. The table underneath shows each model's prediction at your shock and at the four historical months.

In [ ]:
WINDOW = "1990-2007"   # also "1950-2007", "1926-2007" or "2010-2019"
SHOCK = -30.0          # market return in the month, per cent

def show_trap(window=WINDOW, shock=SHOCK):
    models, train = fitted[window]
    A.trap_figure(data, train, models, shock, window).show()
    display(A.trap_table(data, models, shock))

if HAVE_WIDGETS:
    widgets.interact(
        show_trap,
        window=widgets.Dropdown(options=list(A.TRAIN_WINDOWS), value=WINDOW, description="Training"),
        shock=widgets.FloatSlider(value=SHOCK, min=-45, max=15, step=0.5, description="Shock (%)",
                                  continuous_update=False))
else:
    show_trap()

### What to notice

- Trained on 1990–2007, the **forest predicts −9.2%** for banks whatever the market does, and boosting predicts −8.4%. Beyond the training data they stop responding. The line keeps going.
- **Every model misses October 1929 (−33%) and March 2020 (−24%).** Bank stocks fell harder than their normal-times sensitivity implies. Getting the direction right is not the same as getting the amplification right.
- Switch to **1926–2007** and the forest's floor drops to −25.3%, because it has now seen 1929–32. Switch to **2010–2019** and it cannot imagine anything worse than −9.1%. A tree's worst case is a summary of its worst training months, which is why *what you train on* matters as much as *which algorithm*.

### Exercises

**Core 1.** With `WINDOW = "1950-2007"` the training data include October 1987 (market −22.6%, banks −22.5%). Why is the forest's floor only −12.5%? *Hint: look at `min_samples_leaf`.*

**Core 2.** At `SHOCK = -20`, which model comes closest to what banks did in October 1987? Is that a fair test with the default 1990–2007 window?

**Stretch.** Stressed calibration. Refit the linear backbone on the worst 10% of market months only, `data[data["mkt"] <= data["mkt"].quantile(0.10)]`, and compare its prediction for October 1929 with the full-sample line. How much of the gap to −33% does it close?

## Experiment 2: crash months are joint

Sort every month since 1970 by the market's return. Take the **calm** middle half, the **best** tail and the **worst** tail, and in each set compute the average correlation between the 49 industries. The higher it is, the more the industries move together and the less diversification protects you.

There is a catch. Months with big market moves are, by construction, months when the common market factor is loud, and that alone raises measured correlations. The adjusted figures rescale each correlation for the market's larger variance in the tail.

<details>
<summary><b>The maths: average correlation and the Forbes–Rigobon adjustment</b></summary>

For a set of months $\mathcal T$, let $\rho_{ij}$ be the correlation between industries $i$ and $j$ over $\mathcal T$. The average pairwise correlation is

$$\bar\rho = \frac{1}{k(k-1)} \sum_{i\ne j} \rho_{ij}, \qquad k = 49.$$

If every industry loads on the market, conditioning on months where the market's variance is $(1+\delta)$ times its calm-month variance inflates measured correlations. The Forbes–Rigobon-style correction is

$$\rho^{*}_{ij} = \frac{\rho_{ij}}{\sqrt{1 + \delta\,(1-\rho_{ij}^2)}}, \qquad \delta = \frac{\operatorname{Var}(m \mid \mathcal T)}{\operatorname{Var}(m \mid \text{calm})} - 1 .$$

It is an approximation: it treats the market as the only common factor.
</details>

In [ ]:
rets, mkt = A.industries_since_1970()
order = A.calm_order(rets, mkt)

def avg_pairwise_corr(block):
    c = block.corr().values
    n = c.shape[0]
    return (c.sum() - n) / (n * (n - 1))

lo, q25, q75, hi = mkt.quantile([0.10, 0.25, 0.75, 0.90])
for label, mask in [("calm (middle half)", (mkt > q25) & (mkt < q75)),
                    ("best 10%", mkt >= hi), ("worst 10%", mkt <= lo)]:
    print(f"{label:>18}: {int(mask.sum()):3d} months, average correlation {avg_pairwise_corr(rets[mask]):.2f}")

In [ ]:
TAIL = 0.10   # share of months in each tail; try 0.05 or 0.20

def show_tails(tail=TAIL):
    res = A.tail_correlations(rets, mkt, tail)
    A.tails_figure(rets, mkt, tail, res, order).show()
    print(f"Worst {tail:.0%}: {res['crash']:.2f} raw, {res['crash_adjusted']:.2f} adjusted.  "
          f"Best {tail:.0%}: {res['boom']:.2f} raw, {res['boom_adjusted']:.2f} adjusted.  "
          f"Calm: {res['calm']:.2f}.")

if HAVE_WIDGETS:
    widgets.interact(show_tails, tail=widgets.FloatSlider(value=TAIL, min=0.05, max=0.25, step=0.01,
                                                          description="Tail size", readout_format=".0%",
                                                          continuous_update=False))
else:
    show_tails()

### What to notice

- In the worst 10% of months the average correlation is **0.41**, against **0.20** in calm months and 0.27 in the best months. Crashes are more joint than booms.
- **The honest caveat.** Adjust for the louder market and the crash figure falls to **0.23** (best months 0.19). Most of the jump is one common factor getting louder, which Forbes and Rigobon called *interdependence* rather than contagion.
- The lesson for scenario generators is concrete: get the volatility regime of the common factor right and most of the joint tail follows. Generate each industry on its own and you will understate crashes.

### Exercises

**Core.** Set `TAIL = 0.05`. Does the raw gap between crash and calm months widen or narrow? What happens to the adjusted figure?

**Stretch.** Which five industries' correlation with the market rises most from calm months to the worst 10%? Use `rets[mask].corrwith(mkt[mask])` for each set of months and rank the differences. Do they have anything in common?

## Experiment 3: reverse stress test

An ordinary stress test asks: *in this scenario, how much would we lose?* A reverse stress test starts from the loss and asks: *what is the most plausible scenario that would cause it?* The ECB's 2026 exercise asked 110 banks exactly this question for a 300-basis-point fall in their capital ratio.

With quarterly equity and Treasury returns treated as jointly normal, the answer has a closed form. Plausibility is measured by the Mahalanobis distance from the average quarter, so the search returns the breaching scenario that is closest to normal times, given how the two assets have co-moved in the history you choose.

<details>
<summary><b>The maths: the most plausible breaching scenario</b></summary>

Let returns $x \sim N(\mu, \Sigma)$ and portfolio weights $w$. Minimise the squared distance subject to the loss:

$$\min_x\; (x-\mu)^\top \Sigma^{-1} (x-\mu) \quad \text{subject to} \quad w^\top x = L .$$

The Lagrangian gives

$$x^{*} = \mu + \frac{L - w^\top\mu}{w^\top \Sigma w}\, \Sigma w, \qquad d^{*} = \frac{|L - w^\top\mu|}{\sqrt{w^\top \Sigma w}} ,$$

so the distance to the most plausible breach is simply the loss measured in portfolio standard deviations from the mean. For a bivariate normal, the probability of a quarter at least $d$ from the centre is $e^{-d^2/2}$.
</details>

In [ ]:
q = A.quarterly_stock_bond()

def most_plausible(mu, S, w, L):
    return mu + (L - w @ mu) * (S @ w) / (w @ S @ w)

w = np.array([0.6, 0.4])
for start, end in [(2000, 2021), (1970, 1999)]:
    est, mu, S = A.regime(q, start, end)
    xs = most_plausible(mu, S, w, -10.0)
    d22 = A.distance(q.loc["2022Q2"].values, mu, S)
    print(f"{start}-{end}: correlation {est.corr().iloc[0, 1]:+.2f}; most plausible -10% quarter: "
          f"equities {xs[0]:.1f}%, Treasuries {xs[1]:+.1f}%; "
          f"2022 Q2 has probability {A.tail_probability(d22):.4f} under this model")

In [ ]:
YEARS = (2000, 2021)   # history the model learns from; try (1970, 1999)
LOSS = -10.0           # portfolio loss to reach, per cent per quarter
EQUITY = 0.6           # equity weight; Treasuries get the rest

def show_rst(years=YEARS, loss=LOSS, equity=EQUITY):
    fig, xs, d = A.rst_figure(q, years[0], years[1], equity, loss)
    fig.show()
    print(f"Distance {d:.2f}: a quarter at least this unusual has probability "
          f"{A.tail_probability(d):.4f} under this model.")

if HAVE_WIDGETS:
    widgets.interact(
        show_rst,
        years=widgets.IntRangeSlider(value=YEARS, min=1926, max=2025, description="Years",
                                     continuous_update=False),
        loss=widgets.FloatSlider(value=LOSS, min=-25, max=-2, step=0.5, description="Loss (%)",
                                 continuous_update=False),
        equity=widgets.FloatSlider(value=EQUITY, min=0.0, max=1.0, step=0.05, description="Equity",
                                   continuous_update=False))
else:
    show_rst()

### What to notice

- Learning from **2000–21**, the most plausible way for a 60/40 portfolio to lose 10% in a quarter is an equity fall of 18.8% while Treasuries **rally** 3.1%. Learning from **1970–99**, it is equities −12.7% and Treasuries −5.9%.
- **2022 Q2** (equities −16.2%, Treasuries −3.8%) has probability about 0.3% under the 2000–21 model and about 6% under the 1970–99 model. A reverse stress test inherits the regime it was estimated in.
- Set the equity weight to 100%: the most plausible scenario still has Treasuries rallying (+6.7%), even though you no longer own any. The search returns the most likely state of the world, not just of your portfolio.

### Exercises

**Core.** Learn from the pooled history, `YEARS = (1970, 2021)`. What does the most plausible scenario look like now, and why? *Hint: check the correlation.*

**Stretch.** Replace the single normal distribution with a two-regime mixture, `sklearn.mixture.GaussianMixture(n_components=2, random_state=0)`, fitted on 1970–2021. Draw 200,000 scenarios, keep those where the 60/40 portfolio loses at least 10%, and report the most likely one under the mixture (`score_samples`). How does it compare with the closed form, and which regime does it come from?

## Three lessons

1. **Trees cannot imagine a worse month than they have seen**, and linear models extrapolate the direction, not the crisis amplification. Use machine learning where there is data; let structure, stressed calibration or extreme value theory handle the rest.
2. **Crashes are joint**, mostly because the common factor gets louder. A scenario generator has to reproduce that regime, not just each asset's own history.
3. **A reverse stress test is only as good as its plausibility model**, and that model inherits the history it learned from. 2022 was implausible only to models trained on the two decades before it.

Back to the slides: these are the *Translate*, *Generate* and *Search* stages of the AI stress-testing pipeline.